<a href="https://colab.research.google.com/github/jpdeleon/quicklook/blob/main/notebook/single_transit_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-transit (mono- and duotransit) search

TLS needs two or three transits to find a period. Long-period planets often transit only once per TESS sector, so `single_event_search=True` also looks for individual transit-like dips, checks each against step and ramp systematics, and classifies the target as `none`, `mono`, `duo` or `periodic`. Two matching events in stitched sectors form a **duotransit**, whose period is one of the aliases `dT/n`.

Method and outputs: [docs/single-transit.md](https://github.com/jpdeleon/quicklook/blob/main/docs/single-transit.md).

## Setup

On Google Colab, the next cell installs `quicklook` from GitHub. It takes a minute or two the first time, and no runtime restart is needed. Elsewhere it only prints the install command, so it never replaces a local install.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !python -m pip install -q "quicklook-package @ git+https://github.com/jpdeleon/quicklook.git"
else:
    print('Not on Colab. If needed, install with: pip install "quicklook-package @ git+https://github.com/jpdeleon/quicklook.git"')

In [ ]:
from importlib.metadata import version

print("quicklook-package", version("quicklook-package"))

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import Image, display

from quicklook import TessQuickLook

OUTDIR = Path("outputs")  # figures and *_tls.h5 files are written here
OUTDIR.mkdir(exist_ok=True)

## Example 1: a monotransit, TOI-2180 b in sector 48

TOI-2180 b has P ≈ 261 d and transits once in sector 48, 0.25 d after the data start. Expect one ~22 h, ~4 ppt event at SNR ~50, class `mono`, and a minimum period of ~12.4 d (a second transit at a shorter period would have been seen). With a monotransit, panels 5, 7 and 8 show the event SNR, the event with its systematics models, and the periods the data allow.

In [ ]:
mono = TessQuickLook(
    "TOI-2180",
    sector=48,
    single_event_search=True,
    savefig=True,
    outdir=str(OUTDIR),
    overwrite=True,
)
fig = mono.plot_tql()

In [ ]:
r = mono.tls_results
print("class:", r["event_class"], "| events:", r["n_events"])
for t0, dur, depth, snr in zip(
    r["event_t0s"], r["event_durations"], r["event_depths"], r["event_snrs"]
):
    print(f"T0={t0:.3f} BTJD  duration={dur * 24:.1f} h  depth={depth * 1e3:.2f} ppt  SNR={snr:.0f}")
print(f"minimum period allowed by the data: {r['p_min']:.1f} d")

## Example 2: a duotransit, TOI-2180 b in sectors 19 and 48

TOI-2180 b also transits in sector 19, 780.5 d (3 × P) earlier. Stitching the two sectors (a sector list needs a single `exptime`) pairs the two events. Expect matching depths and durations, 31 allowed aliases `780.5/n` with P ≥ 25.2 d, and the true period (n = 3, 260.2 d) as the most likely, closely followed by n = 2. Panels 7 and 8 then show both events overlaid and the aliases with their probabilities.

In [ ]:
duo = TessQuickLook(
    "TOI-2180",
    sector=[19, 48],
    exptime=120,
    single_event_search=True,
    savefig=True,
    outdir=str(OUTDIR),
    overwrite=True,
)
fig = duo.plot_tql()

In [ ]:
pair = duo.duo
print(f"dT = {pair.delta_t:.2f} d, depth ratio {pair.depth_ratio:.2f}, "
      f"duration ratio {pair.duration_ratio:.2f}")
print(f"{len(pair.aliases)} allowed aliases, P_min = {pair.p_min:.1f} d")
for alias in sorted(pair.aliases, key=lambda a: -a.weight)[:4]:
    print(f"  n={alias.n:2d}  P={alias.period:7.2f} d  probability={alias.weight:.0%}")

## Same runs from the command line

```bash
ql run --name TOI-2180 --sector 48 --single-event-search --save
ql run --name TOI-2180 --sectors 19,48 --exptime 120 --single-event-search --save
```

In the web GUI: tick **Single-transit search** under Period Search, with Sector `48`, or Sector `19,48` and Exposure Time `120`.